In [1]:
import os
os.environ['TORCH_HOME'] = './data'
os.environ['NO_PROXY'] = 'pytorch.org,torchvision.models'

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data.dataloader import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class ImprovedAlexNet(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.feature = nn.Sequential(
            # input: 3 * 32 * 32
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(64),  # 添加BatchNorm
            nn.MaxPool2d(kernel_size=2),
            
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(192),  # 添加BatchNorm
            nn.MaxPool2d(kernel_size=2),
            
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(384),
            
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(256),
            
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(kernel_size=2)
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 4 * 4, 1024),  # 减小全连接层大小
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            
            nn.Linear(512, num_classes)
        )
        
    def forward(self, x):
        x = self.feature(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# 数据预处理
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),  # CIFAR-100的统计值
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

# 数据集
train_set = torchvision.datasets.CIFAR100('./data', True, transform_train, download=True)
test_set = torchvision.datasets.CIFAR100('./data', False, transform_test, download=True)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2)

# 模型和优化器
net = ImprovedAlexNet(100).to(device)
criterion = nn.CrossEntropyLoss()

# 改进的优化器设置
optimizer = optim.SGD(net.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0 = 15,T_mult =2 )  # 余弦退火

def train(net, train_loader, criterion, epochs=100):
    net.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for i, data in enumerate(train_loader, 0):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            if i % 100 == 99:
                print(f'Epoch [{epoch+1}/{epochs}], Batch [{i+1}], Loss: {running_loss/100:.4f}, Acc: {100*correct/total:.2f}%')
                running_loss = 0.0
        
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]
        epoch_acc = 100 * correct / total
        print(f'Epoch {epoch+1} finished. LR: {current_lr:.6f}, Train Acc: {epoch_acc:.2f}%')
    
    print('Finished Training')

def test(net, testloader):
    net.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data in testloader:
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = net(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    print(f'Accuracy of the network on the 10000 test images: {accuracy:.2f}%')
    return accuracy

# 训练更多epochs
train(net, train_loader, criterion, 100)
test(net, test_loader)

Epoch [1/100], Batch [100], Loss: 4.1968, Acc: 6.53%
Epoch [1/100], Batch [200], Loss: 3.7782, Acc: 9.02%
Epoch [1/100], Batch [300], Loss: 3.5238, Acc: 11.35%
Epoch 1 finished. LR: 0.009891, Train Acc: 13.07%
Epoch [2/100], Batch [100], Loss: 3.2129, Acc: 20.48%
Epoch [2/100], Batch [200], Loss: 3.1173, Acc: 21.59%
Epoch [2/100], Batch [300], Loss: 2.9916, Acc: 22.86%
Epoch 2 finished. LR: 0.009568, Train Acc: 23.74%
Epoch [3/100], Batch [100], Loss: 2.7227, Acc: 30.22%
Epoch [3/100], Batch [200], Loss: 2.6914, Acc: 30.58%
Epoch [3/100], Batch [300], Loss: 2.6616, Acc: 31.12%
Epoch 3 finished. LR: 0.009045, Train Acc: 31.83%
Epoch [4/100], Batch [100], Loss: 2.4320, Acc: 36.24%
Epoch [4/100], Batch [200], Loss: 2.3913, Acc: 36.54%
Epoch [4/100], Batch [300], Loss: 2.3733, Acc: 36.69%
Epoch 4 finished. LR: 0.008346, Train Acc: 37.25%
Epoch [5/100], Batch [100], Loss: 2.2000, Acc: 40.93%
Epoch [5/100], Batch [200], Loss: 2.2034, Acc: 41.18%
Epoch [5/100], Batch [300], Loss: 2.1828, Acc:

KeyboardInterrupt: 